## This is the code for generating Feature Heatmap plot.

By default, if you play this file directly, it will generate the Feature Heatmap plot with respect to our experiment result.

**Guideline**:  
Read in the Influence lists -> Compute the pairwise weighted Kendall tau matrix -> Turn the Matrix into the corresponding heatmap

**Format**:  
**Input** The Influence lists that you read in.  
**Output**  The Feature Heatmap Plot 

You don't need to change anything else if you only want to produce the Heatmap plot. You only need to change the read_csv part to the new data that you generated in the estimation code.

In [26]:
import pandas as pd
from scipy.stats import kendalltau,weightedtau
import numpy as np
import matplotlib.pyplot as plt
import dcor
import seaborn as sns

# Influence Ranking Read-In Area

Again, change here for your desired influence lists.

In [27]:
IF_1 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_00.csv")
IF_2 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_10.csv")
IF_3 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_14.csv")
IF_4 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_18.csv")
IF_5 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_22.csv")
IF_6 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_26.csv")
IF_7 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_30.csv")
IF_8 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_34.csv")
IF_9 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_38.csv")
IF_10 = pd.read_csv("FeatureRun5/IF_Feature_Column_10_42.csv")

TC_1 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_00.csv")
TC_2 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_10.csv")
TC_3 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_14.csv")
TC_4 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_18.csv")
TC_5 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_22.csv")
TC_6 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_26.csv")
TC_7 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_30.csv")
TC_8 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_34.csv")
TC_9 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_38.csv")
TC_10 = pd.read_csv("FeatureRun5/TC_Feature_Column_10_42.csv")

In [28]:
sorted_IF_lists = [IF_1, IF_2,IF_3,IF_4,IF_5,IF_6,IF_7,IF_8,IF_9,IF_10]
sorted_TC_lists = [TC_1, TC_2,TC_3,TC_4,TC_5,TC_6,TC_7,TC_8,TC_9,TC_10]
Train_Size = [10,20,24,28,32,36,40,44,48,52]

1. Here we initial the weighted kendall tau matrix.

In [29]:
num_lists = len(sorted_IF_lists)

In [30]:
tau_matrix_TC = np.full((num_lists, num_lists), np.nan)
tau_matrix_IF = np.full((num_lists, num_lists), np.nan)
print(tau_matrix_IF)

[[nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]]


2. Here we compute the pairwise weighted kendall tau value for all the Influence Function's ranked influence lists.

In [31]:
for i in range(num_lists):
    tau_matrix_IF[i, i] = 1.0
    for j in range(i + 1, num_lists):
        df1 = sorted_IF_lists[i]
        df2 = sorted_IF_lists[j]
        max_abs1 = df1['Score'].abs().max()
        df1['Score'] = df1['Score'] / max_abs1    
        max_abs2 = df2['Score'].abs().max()
        df2['Score'] = df2['Score'] / max_abs2
        merged = pd.merge(df1, df2, on='Train_ID', how='inner')
        wt, _ = weightedtau(merged["Score_x"], merged["Score_y"],
                        weigher = lambda x: (np.abs(merged["Score_x"][x]) + np.abs(merged["Score_y"][x]))/2, rank = None)
        #wt, _ = kendalltau(merged["Score_x"], merged["Score_y"])
        tau_matrix_IF[i, j] = wt
        tau_matrix_IF[j, i] = wt

print(tau_matrix_IF)

[[1.         0.80146409 0.76707034 0.75673716 0.75228318 0.74051953
  0.72607694 0.71535795 0.71291124 0.70689379]
 [0.80146409 1.         0.90171921 0.86870469 0.83568622 0.8169459
  0.79956698 0.77326766 0.76992281 0.75379581]
 [0.76707034 0.90171921 1.         0.92351286 0.87506489 0.8517017
  0.83420651 0.8033937  0.79049073 0.77541039]
 [0.75673716 0.86870469 0.92351286 1.         0.90115963 0.87667419
  0.84813536 0.81861662 0.80731408 0.78816283]
 [0.75228318 0.83568622 0.87506489 0.90115963 1.         0.92883252
  0.89689471 0.86430553 0.84430528 0.83130166]
 [0.74051953 0.8169459  0.8517017  0.87667419 0.92883252 1.
  0.93025786 0.88703797 0.86028037 0.84579788]
 [0.72607694 0.79956698 0.83420651 0.84813536 0.89689471 0.93025786
  1.         0.9176319  0.88260491 0.86345125]
 [0.71535795 0.77326766 0.8033937  0.81861662 0.86430553 0.88703797
  0.9176319  1.         0.93055463 0.89928365]
 [0.71291124 0.76992281 0.79049073 0.80731408 0.84430528 0.86028037
  0.88260491 0.9305546

3. Here we compute the pairwise weighted kendall tau value for all the TracIn's ranked influence lists.

In [32]:
for i in range(num_lists):
    tau_matrix_TC[i, i] = 1.0
    for j in range(i + 1, num_lists):
        df1 = sorted_TC_lists[i]
        df2 = sorted_TC_lists[j]
        max_abs1 = df1['Score'].abs().max()
        df1['Score'] = df1['Score'] / max_abs1    
        max_abs2 = df2['Score'].abs().max()
        df2['Score'] = df2['Score'] / max_abs2
        merged = pd.merge(df1, df2, on='Train_ID', how='inner')
        wt, _ = weightedtau(merged["Score_x"], merged["Score_y"],
                         weigher = lambda x: (np.abs(merged["Score_x"][x]) + np.abs(merged["Score_y"][x]))/2, rank = None)
        #wt, _ = kendalltau(merged["Score_x"], merged["Score_y"])
        tau_matrix_TC[i, j] = wt
        tau_matrix_TC[j, i] = wt

print(tau_matrix_TC)

[[1.         0.95329282 0.94930773 0.94543866 0.94229152 0.94083072
  0.93866991 0.93611992 0.93458655 0.93181939]
 [0.95329282 1.         0.97565001 0.96588601 0.95805128 0.95457111
  0.9509532  0.94517928 0.94228205 0.93987993]
 [0.94930773 0.97565001 1.         0.97658992 0.96612415 0.96100451
  0.95718914 0.95073163 0.94743822 0.94471052]
 [0.94543866 0.96588601 0.97658992 1.         0.97449123 0.96652163
  0.96238916 0.95525068 0.95183066 0.94880054]
 [0.94229152 0.95805128 0.96612415 0.97449123 1.         0.97842237
  0.97155251 0.96327783 0.95898027 0.9557192 ]
 [0.94083072 0.95457111 0.96100451 0.96652163 0.97842237 1.
  0.97980398 0.96898885 0.96335471 0.959574  ]
 [0.93866991 0.9509532  0.95718914 0.96238916 0.97155251 0.97980398
  1.         0.97712106 0.97040883 0.96563321]
 [0.93611992 0.94517928 0.95073163 0.95525068 0.96327783 0.96898885
  0.97712106 1.         0.98154889 0.97338396]
 [0.93458655 0.94228205 0.94743822 0.95183066 0.95898027 0.96335471
  0.97040883 0.98154

4. Now we turn each weighted tau matrix into a triangular heatmap. This is to remove the duplicate half.

In [33]:
labels = [f"{size:}" for idx, size in enumerate(Train_Size)]
df_tau_IF = pd.DataFrame(tau_matrix_IF, index=labels, columns=labels)
df_tau_TC = pd.DataFrame(tau_matrix_TC, index=labels, columns=labels)

In [34]:
run_number = 5

In [35]:
df_tau_IF.to_csv(
    f"FOIF_TauMatrix_Run{run_number}.csv"
)

df_tau_TC.to_csv(
    f"TracIn_TauMatrix_Run{run_number}.csv"
)